# Hands-free phone agent: full pipeline on a free GPU

Use this when your laptop is too weak. **Runtime > Change runtime type > T4 GPU**, then run the cells top to bottom.

1. Get the code
2. *(optional)* Model council: bigger local models teach and check each other
3. Build the training data
4. Train the small phone model
5. Shrink it for the phone (GGUF)
6. Test it on the simulated phone
7. Download the phone model

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
# The repo is private: paste a GitHub token with read access (github.com/settings/tokens).
from getpass import getpass
import os
token = getpass("GitHub token: ")
!git clone -q https://{token}@github.com/Mo3gza21/handsfree-phone-agent.git
os.chdir("handsfree-phone-agent")
!pip install -q -r requirements.txt

## 2. Model council *(optional, ~1-3 hours)*

Colab's GPU can run bigger teachers than a weak laptop. Put the models you want in `MODELS`
(any names from ollama.com/library). To use the exact GGUF files from your laptop instead, upload them to
Google Drive and set `GGUF_FILES`. Skip this section to train on simulator data only, which already works well.

In [ ]:
MODELS = ["qwen2.5:7b", "llama3.1:8b", "gemma2:9b"]   # teachers that talk to each other
GGUF_FILES = []   # e.g. ["/content/drive/MyDrive/models/my-model.gguf"]

if GGUF_FILES:
    from google.colab import drive
    drive.mount("/content/drive")
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null
import subprocess, time, pathlib
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)
for f in GGUF_FILES:
    name = pathlib.Path(f).stem.lower().replace(" ", "-")
    pathlib.Path("Modelfile").write_text(f"FROM {f}\n")
    !ollama create {name} -f Modelfile
    MODELS.append(name)
for m in MODELS:
    if ":" in m:
        !ollama pull {m} > /dev/null
!ollama list

In [ ]:
models = " ".join(MODELS)
# Who is the best teacher? Every model drives the simulated phone and gets scored.
!cd pipeline && python council.py leaderboard --models {models} --n 60 --workers 4

In [ ]:
import json
best = json.load(open("data/leaderboard.json"))["ranking"]
print("Best teacher first:", best)
ordered = " ".join(best)
# Best teacher rewrites requests the way people really talk; the others check meaning.
!cd pipeline && python council.py paraphrase --models {ordered} --n 400 --k 3 --workers 4
# Teachers invent tasks for apps the simulator lacks; the others must agree on every step.
!cd pipeline && python council.py invent --models {ordered} --n 300

## 3. Build the training data

In [ ]:
import os
extra = " ".join(p for p in ["../data/paraphrased.jsonl", "../data/invented.jsonl"] if os.path.exists(p[1:]))
!cd pipeline && python build_dataset.py --sim 5000 --extra {extra}

## 4. Train (~30-60 min on a T4)

`Qwen/Qwen2.5-1.5B-Instruct` gives a ~1 GB phone model and handles Arabic and English. For older
phones use `Qwen/Qwen2.5-0.5B-Instruct` (~400 MB).

In [ ]:
BASE = "Qwen/Qwen2.5-1.5B-Instruct"
!python training/train_lora.py --base {BASE} --batch 8 --grad-accum 2 --epochs 2

## 5. Shrink it for the phone

In [ ]:
!python training/export_gguf.py --quant Q4_K_M
!ls -lh outputs/*.gguf

## 6. Test it on the simulated phone

Runs the trained model on 150 tasks it has never seen. If some task types are weak, the last cell
builds new data focused on them; then rerun steps 4-6. That is the self-improvement loop.

In [ ]:
import subprocess, time, glob
server = next(p for p in glob.glob("outputs/llama.cpp/build/bin/**/llama-server", recursive=True))
srv = subprocess.Popen([server, "-m", "outputs/phone-agent-q4_k_m.gguf", "--port", "8080", "-ngl", "99", "-c", "2048"],
                       stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(20)
!cd pipeline && python council.py leaderboard --models student@http://localhost:8080/v1 --n 150 --workers 4 --out ../data/student_board.json
srv.terminate()

In [ ]:
# Optional: next round, focused on what the student got wrong. Then rerun steps 4-6.
!cd pipeline && python build_dataset.py --sim 5000 --focus ../data/student_board.json --focus-model student --extra {extra}

## 7. Download the phone model

In [ ]:
from google.colab import files
files.download("outputs/phone-agent-q4_k_m.gguf")